# Pandas Interview Dataset Setup

Run the cells below to install packages, load and inspect the bundled realistic datasets. Keep `data/` alongside the notebooks.

> **Realistic Dataset v2:** Shared across Python, Pandas, and PySpark. Existing columns retained. Synthetic data includes intentional dirty records and malformed JSON for interview exercises.


In [ ]:
# Run once in your VS Code venv / Jupyter kernel.
%pip install -q 'pandas>=2.2,<4' 'numpy>=1.26'


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Keep this notebook and the data/ folder together.
DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('pandas_de_practice/data')
assert DATA_DIR.exists(), 'Cannot find data/. Open the extracted pandas_de_practice folder in VS Code.'

customers = pd.read_csv(DATA_DIR / 'customers.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
transactions = pd.read_csv(DATA_DIR / 'transactions.csv')
pipeline_logs = pd.read_csv(DATA_DIR / 'pipeline_logs.csv')
cdc_changes = pd.read_csv(DATA_DIR / 'cdc_changes.csv')
events = pd.read_json(DATA_DIR / 'events.jsonl', lines=True)
events_duplicates = pd.read_json(DATA_DIR / 'events_duplicates.jsonl', lines=True)
customers_dirty = pd.read_json(DATA_DIR / 'customers_dirty.json')

print('Loaded:', {k:len(globals()[k]) for k in ['customers','orders','order_items','products','transactions','pipeline_logs','cdc_changes','events','events_duplicates','customers_dirty']})


In [ ]:
for name in ['customers','orders','order_items','products','transactions','pipeline_logs','cdc_changes','events','events_duplicates','customers_dirty']:
    df = globals()[name]
    print(f'{name:20s} {len(df):>5} rows | columns: {list(df.columns)}')


In [ ]:
import csv
import json
from pathlib import Path

# Adjust this path if your data folder is elsewhere
DATA = Path("data")

csv_path = DATA / "customers.csv"
json_path = DATA / "customers.json"
jsonl_path = DATA / "customers.jsonl"

# Load existing customer records
with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    customers = list(reader)

# Select an existing customer with a valid email.
# Preserve existing blank and whitespace-only test cases.
target = next(
    c for c in customers
    if c.get("email") and c["email"].strip()
)

target_id = target["customer_id"]

# Set email to NULL
target["email"] = ""

# Update CSV
with csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(customers)

# Update JSON and JSONL, preserving existing field types
with json_path.open("r", encoding="utf-8") as f:
    customers_json = json.load(f)

for c in customers_json:
    if str(c["customer_id"]) == str(target_id):
        c["email"] = None

with json_path.open("w", encoding="utf-8") as f:
    json.dump(customers_json, f, indent=2, ensure_ascii=False)

with jsonl_path.open("w", encoding="utf-8") as f:
    for c in customers_json:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"Updated customer_id: {target_id}")
print("CSV, JSON and JSONL updated successfully.")